# Does the embedding know the scene?

Appendix A removes the **person** from the evaluation but not the **scene**: an episode is one
recording session (studio, lighting, cameras, audio chain, day), and 82–94% of the holdout clips still
come from an episode that appears in training. This notebook measures how much of that the frozen
DINOv2 embedding carries.

| probe | question | folds |
|---|---|---|
| episode, subjects held out | can the episode of a *new* person be named? | grouped by subject |
| speaker, random folds | how much identity sits in the same vector? | random |
| episode of one speaker | same face across many episodes: is the scene separable? | random |

Every probe is also run on permuted labels, which gives the empirical chance level for the same
classifier, dimensionality and folds. Probes 2–3 use random folds on purpose: they ask whether the
vector *separates* the classes, so their accuracy is an upper bound (clips of the same round are
adjacent seconds of the same video). Probe 1 is the conservative one.

`Runtime → Run all`. Most of the wall clock is reading the cache, not training.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# ------------------------------------------------------------------ paths
import os, shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
CACHE_HINT = ""        # paste an explicit path here if the search below finds nothing
COPY_TO_LOCAL = True   # one bulk copy is much faster than 1,600 single reads from Drive


def find_cache(root: Path, name: str = "dinov2-small", max_depth: int = 8):
    root = root.resolve()
    for dirpath, dirnames, _ in os.walk(root):
        if len(Path(dirpath).relative_to(root).parts) > max_depth:
            dirnames[:] = []
            continue
        if name in dirnames and any((Path(dirpath) / name).glob("*.npy")):
            return Path(dirpath) / name
        dirnames[:] = [d for d in dirnames if d not in {"clips", "videos", ".git"}]
    return None


cache = Path(CACHE_HINT).expanduser() if CACHE_HINT else find_cache(DRIVE_ROOT)
assert cache and cache.exists(), "cache not found: set CACHE_HINT to the folder with one .npy per clip"
dataset = cache.parent.parent / "dataset.csv"
assert dataset.exists(), f"dataset.csv not found at {dataset}"

print(f"cache    {cache}  ({len(list(cache.glob('*.npy')))} vectors)")
print(f"dataset  {dataset}")

if COPY_TO_LOCAL and str(cache).startswith("/content/drive"):
    local = Path("/content/cache") / cache.name
    if not local.exists():
        print("copying the cache to local disk ...")
        shutil.copytree(cache, local)
    cache = local
    print(f"reading from {cache}")

cache    /content/drive/MyDrive/dolos/cached_features/dinov2-small  (1299 vectors)
dataset  /content/drive/MyDrive/dolos/dataset.csv
copying the cache to local disk ...
reading from /content/cache/dinov2-small


In [3]:
# ------------------------------------------------------------------ probes
import csv
import re
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler

FOLDS, SEED = 5, 0
MIN_EPISODES, MIN_CLIPS = 10, 40


def read_meta(path):
    """clip_id -> (subject, episode); the episode is the EP<n> token of the source name."""
    out = {}
    with open(path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            name = row["source_name"]
            match = re.search(r"ep\s*(\d+)", name, re.IGNORECASE)
            episode = f"EP{int(match.group(1))}" if match else name.split("_")[0]
            out[row["clip_id"].lower()] = (row["subject"], episode)
    return out


def read_vectors(cache, meta):
    x, ids, missing = [], [], 0
    for clip_id in meta:
        path = cache / f"{clip_id}.npy"
        if not path.exists():
            missing += 1
            continue
        x.append(np.load(path).mean(axis=0))
        ids.append(clip_id)
    print(f"{len(ids)} vectors read, {missing} clips without a cache file")
    return np.vstack(x), ids


def folds_for(n, groups):
    if groups is None:
        return list(KFold(n_splits=min(FOLDS, n), shuffle=True, random_state=SEED).split(np.zeros(n)))
    unique = len(set(groups))
    return list(GroupKFold(n_splits=max(2, min(FOLDS, unique))).split(np.zeros(n), groups=groups))


def accuracy(x, y, groups=None):
    x = StandardScaler().fit_transform(x)
    hit = total = 0
    for train, test in folds_for(len(y), groups):
        if len(set(y[train])) < 2:
            continue
        model = LogisticRegression(max_iter=2000, C=1.0).fit(x[train], y[train])
        hit += int((model.predict(x[test]) == y[test]).sum())
        total += len(test)
    return hit / total


def permuted(x, y, groups=None, repeats=5):
    rng = np.random.default_rng(SEED)
    values = [accuracy(x, rng.permutation(y), groups) for _ in range(repeats)]
    return float(np.mean(values)), float(np.std(values))


ROWS = []


def probe(name, x, y, groups=None):
    y = np.asarray(y)
    acc = accuracy(x, y, groups)
    chance, spread = permuted(x, y, groups)
    _, counts = np.unique(y, return_counts=True)
    ROWS.append(dict(probe=name, accuracy=acc, majority=counts.max() / len(y),
                     permuted=chance, permuted_sd=spread, classes=len(counts), n=len(y)))
    print(f"{name:<32} acc {acc:6.3f}  majority {counts.max() / len(y):6.3f}  "
          f"permuted {chance:6.3f} ± {spread:5.3f}  classes {len(counts):3d}  n {len(y)}")

In [4]:
# ------------------------------------------------------------------ run
meta = read_meta(dataset)
x, ids = read_vectors(cache, meta)
subjects = np.array([meta[i][0] for i in ids])
episodes = np.array([meta[i][1] for i in ids])
print(f"episodes {len(set(episodes))}   subjects {len(set(subjects))}\n")

probe("episode, subjects held out", x, episodes, subjects)
probe("speaker, random folds", x, subjects)
probe("episode, random folds", x, episodes)

seen = {}
for clip_id in ids:
    subject, episode = meta[clip_id]
    seen.setdefault(subject, set()).add(episode)

frequent = sorted(
    (s for s in seen if len(seen[s]) >= MIN_EPISODES
     and sum(1 for c in ids if meta[c][0] == s) >= MIN_CLIPS),
    key=lambda s: -len(seen[s]))

for speaker in frequent:
    rows = [k for k, c in enumerate(ids) if meta[c][0] == speaker]
    probe(f"episode of {speaker[:20]}", x[rows], [meta[ids[k]][1] for k in rows])
    print(f"{'':32} {len(rows)} clips over {len(seen[speaker])} episodes, same face every time")

1299 vectors read, 379 clips without a cache file
episodes 59   subjects 145

episode, subjects held out       acc  0.074  majority  0.028  permuted  0.017 ± 0.004  classes  59  n 1299
speaker, random folds            acc  0.963  majority  0.143  permuted  0.068 ± 0.006  classes 145  n 1299
episode, random folds            acc  0.880  majority  0.028  permuted  0.017 ± 0.002  classes  59  n 1299
episode of Lee Mack              acc  0.871  majority  0.043  permuted  0.018 ± 0.007  classes  45  n 186
                                 186 clips over 45 episodes, same face every time
episode of David Mitchell        acc  0.814  majority  0.076  permuted  0.040 ± 0.006  classes  42  n 172
                                 172 clips over 42 episodes, same face every time


In [5]:
# ------------------------------------------------------------------ table
import pandas as pd

out = pd.DataFrame(ROWS)
out["delta_pp"] = 100 * (out["accuracy"] - out["permuted"])
print(out.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
target = Path("/content/episode_probe.csv") if Path("/content").exists() else Path("episode_probe.csv")
out.to_csv(target, index=False)
print(f"\nsaved {target.resolve()}")

                     probe  accuracy  majority  permuted  permuted_sd  classes    n  delta_pp
episode, subjects held out     0.074     0.028     0.017        0.004       59 1299     5.712
     speaker, random folds     0.963     0.143     0.068        0.006      145 1299    89.469
     episode, random folds     0.880     0.028     0.017        0.002       59 1299    86.282
       episode of Lee Mack     0.871     0.043     0.018        0.007       45  186    85.269
 episode of David Mitchell     0.814     0.076     0.040        0.006       42  172    77.442

saved /content/episode_probe.csv


## How to read it

- **episode, subjects held out** is the number the appendix needs. Chance is the permuted column; if
  accuracy sits at chance, the subject split is not silently leaking the scene, and the 56% stands as
  published. If it is well above chance, the embedding names the episode for people it has never seen,
  which is the mechanism the appendix currently only hypothesises.
- **speaker, random folds** is the reference: it measures how much of the vector is identity. Episode
  accuracy should be read against it — if the two are similar, the episode probe is mostly reading the
  person again, not the room.
- **episode of one speaker** is the clean test: the face never changes, so any accuracy above chance is
  the studio, the lights and the day. Treat it as an upper bound, because clips of the same round are
  adjacent seconds of the same video.

Quote the `permuted` mean ± sd next to each accuracy; a probe that does not clear its own null by a
few standard deviations is not a finding.